# MaskWall Step 4: fine-tune MiniLM and DeBERTa-v3-small (Colab GPU)

Fine-tunes two encoders as binary prompt-injection/jailbreak classifiers on the **same splits** as the baselines
(`ml/train_baseline.py`) and evaluates them **exactly the same way**, by importing the baseline's own evaluation
functions rather than copying them.

**Status: prepared but not yet run.** Time per model is an estimate (roughly 20 to 60 min on a T4), not a measurement.

## Before you run (about 5 minutes)
1. **Runtime → Change runtime type → T4 GPU.** The notebook stops if there is no GPU.
2. Have these 8 files from your local repo ready to upload in step 3 (the processed splits are gitignored, so they are *not* on GitHub):
   - `ml/data/processed/train.parquet`, `train_augmented.parquet`, `indirect_eval.parquet`, `val.parquet`, `test.parquet`, `xstest_safe.parquet`
   - (`train_augmented.parquet` and `indirect_eval.parquet` come from `python ml/data/load_datasets.py --augment-indirect`)
   - `ml/data/tricky.csv`
   - `ml/train_baseline.py` (supplies the shared evaluation code)
3. Run cells top to bottom. The upload cell opens a file picker; select all 8 files at once.
4. Optional: set `USE_DRIVE = True` to save checkpoints to Google Drive. Recommended, because DeBERTa-small is about 570 MB and Step 5 (ONNX export) needs the checkpoints.
5. When finished, download `metrics_transformers.json` and `transformer_val_errors.md` into `ml/results/` locally.

## What is fixed (decided before seeing any result)
| | |
|---|---|
| Models | `sentence-transformers/all-MiniLM-L6-v2` (22M, same encoder as baseline 4) and `microsoft/deberta-v3-small` (141M) |
| Truncation | **head + tail**: first 256 + last 254 content tokens, so `[CLS]` + 510 + `[SEP]` = 512 (the position limit). Long jailbreaks often hide the payload at the end. |
| Epochs | up to 4; stop early if val average precision did not improve, but never before epoch 3 |
| Selection | best epoch by **val average precision** (threshold-free). Val only. |
| Threshold | maximises **val F1** (ties → higher), plus a second reported threshold at val FPR ≤ 1% (reporting only) |
| Seed | 42 for python, numpy and torch. GPU kernels are not bit-exact, so reruns can differ slightly. |
| Training sets | `cleaned` labels, each trained **without** (`train.parquet`) and **with** (`train_augmented.parquet`, +800 synthetic indirect-injection rows) augmentation, so the effect is visible. Set `RUN_PLAIN` / `RUN_AUGMENTED` to `False` to skip one (each variant is about 2 models x 20 to 60 min). |
| Labels | `cleaned` by default. Set `RUN_ORIGINAL_LABELS = True` to also train on the published labels (doubles the time). |
| Test set | evaluated **once per model, after** the threshold is frozen. Do not rerun cells after looking at test numbers to "improve" them. |

Not done here: ONNX export, quantization and latency (Step 5).

## 1. Setup

In [ ]:
!pip -q install "transformers>=4.44" accelerate sentencepiece tiktoken protobuf scikit-learn pandas pyarrow joblib

In [ ]:
import json, os, platform, random, sys, time
import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

# ------------------------------------------------------------------ config (fixed before any result)
SEED = 42
MODELS = {
    "minilm":           dict(hf="sentence-transformers/all-MiniLM-L6-v2", lr=5e-5, batch=32),
    "deberta_v3_small": dict(hf="microsoft/deberta-v3-small",              lr=3e-5, batch=16),
}
HEAD, TAIL = 256, 254          # content tokens; +[CLS] +[SEP] = 512
MAX_EPOCHS, MIN_EPOCHS, PATIENCE = 4, 3, 1
WARMUP_FRAC, WEIGHT_DECAY, GRAD_CLIP = 0.1, 0.01, 1.0
FP16 = True
RUN_ORIGINAL_LABELS = False    # True: also train/evaluate on the published labels (label_original)
USE_DRIVE = False              # True: save checkpoints to Google Drive

RUN_PLAIN = True               # train on train.parquet
RUN_AUGMENTED = True           # train on train_augmented.parquet (train + synthetic indirect injections)
# variant -> (label column, train file); val, test, tricky, xstest are the same for every variant
VARIANTS = {}
for _lab, _col in [("cleaned", "label")] + ([("original_labels", "label_original")] if RUN_ORIGINAL_LABELS else []):
    if RUN_PLAIN:
        VARIANTS[_lab] = (_col, "train")
    if RUN_AUGMENTED:
        VARIANTS[_lab + "_aug"] = (_col, "train_augmented")

ROOT = "/content/maskwall"
OUT = "/content/out"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/maskwall_ckpt"
os.makedirs(f"{ROOT}/data/processed", exist_ok=True)
os.makedirs(OUT, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## 2. Upload the 8 files

In [ ]:
from google.colab import files

uploaded = files.upload()   # select all 8 files at once
for name, content in uploaded.items():
    if name.endswith(".parquet"):
        dest = f"{ROOT}/data/processed/{name}"
    elif name == "tricky.csv":
        dest = f"{ROOT}/data/{name}"
    elif name == "train_baseline.py":
        dest = f"{ROOT}/{name}"
    else:
        print("ignoring", name); continue
    open(dest, "wb").write(content)
    print("saved", dest)

sys.path.insert(0, ROOT)
import train_baseline as tb     # shared evaluation: same code path as the baselines

## 3. Load the data and check it

In [ ]:
raw = {s: pd.read_parquet(f"{ROOT}/data/processed/{s}.parquet") for s in ["train", "train_augmented", "val", "test", "xstest_safe", "indirect_eval"]}
tricky = pd.read_csv(f"{ROOT}/data/tricky.csv")

for s, d in raw.items():
    assert {"text", "label", "source", "label_original"} <= set(d.columns), f"{s}: columns {list(d.columns)}"
    print(f"{s:12s} {len(d):5d} rows, {int(d.label.sum()):4d} attacks")
print(f"{'tricky':12s} {len(tricky):5d} rows")

# Sanity: no exact text overlap between train/val/test (the loader asserts this too).
norm = lambda t: " ".join(t.split()).lower()
sets = {s: set(map(norm, raw[s].text)) for s in ["train", "train_augmented", "val", "test"]}
assert not (sets["train_augmented"] & sets["val"]) and not (sets["train_augmented"] & sets["test"])
ie_set = set(map(norm, raw["indirect_eval"].text))   # held-out indirect set: evaluation only
assert not (ie_set & sets["train"]) and not (ie_set & sets["train_augmented"])
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
print("no exact overlap between splits")

## 4. Head + tail tokenization

Long prompts are cut in the middle, not at the end: the first 256 and last 254 content tokens are kept. A plain
"first 512 tokens" cut would drop the end of long jailbreaks, where the actual request often sits.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


def encode_head_tail(tok, texts):
    """Token ids with special tokens, head+tail truncated to at most HEAD+TAIL+2 = 512 tokens."""
    # Both models use the BERT-style layout [CLS] x [SEP]. Built by hand (not via tokenizer helpers whose
    # names differ between transformers versions).
    assert tok.cls_token_id is not None and tok.sep_token_id is not None
    ids_all = tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    out = []
    for ids in ids_all:
        if len(ids) > HEAD + TAIL:
            ids = ids[:HEAD] + ids[-TAIL:]
        out.append([tok.cls_token_id] + ids + [tok.sep_token_id])
    return out


def collate(seqs, pad_id):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        att[i, :len(s)] = 1
    return ids, att


@torch.no_grad()
def predict(model, enc, pad_id, bs=64):
    """P(attack) for each encoded text, in the original order."""
    model.eval()
    order = np.argsort([len(x) for x in enc])      # length-sorted batches = less padding
    probs = np.zeros(len(enc))
    for i in range(0, len(enc), bs):
        idx = order[i:i + bs]
        ids, att = collate([enc[j] for j in idx], pad_id)
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            logits = model(input_ids=ids.cuda(), attention_mask=att.cuda()).logits
        probs[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return probs

## 5. Training loop

A plain PyTorch loop instead of `Trainer`, so nothing depends on library version changes and every step is visible:
AdamW, linear warmup then decay, fp16 autocast with a grad scaler, gradient clipping. No class weighting: the baselines
don't use any either, and the val-tuned threshold handles the 3:1 imbalance.
**Epoch selection uses val only** (average precision). Test is not touched in this cell.

In [ ]:
from sklearn.metrics import average_precision_score


def train_one(key, cfg, data, label_col):
    """Fine-tune one model. Returns (model, tokenizer, encodings, history, best_epoch, seconds)."""
    set_seed(SEED)
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    tok.model_max_length = 10**6                        # we truncate ourselves
    model = AutoModelForSequenceClassification.from_pretrained(cfg["hf"], num_labels=2).cuda()
    enc = {s: encode_head_tail(tok, d["text"]) for s, d in data.items()}
    n_tr = int(len(enc["train"]))
    y_tr = data["train"]["label"].to_numpy()
    y_va = data["val"]["label"].to_numpy()

    steps_per_epoch = int(np.ceil(n_tr / cfg["batch"]))
    total = steps_per_epoch * MAX_EPOCHS
    no_decay = ["bias", "LayerNorm.weight", "layernorm", "LayerNorm.bias"]
    params = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(params, lr=cfg["lr"])
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)

    history, best_ap, best_state, best_epoch, stale = [], -1.0, None, 0, 0
    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = np.random.default_rng(SEED + epoch).permutation(n_tr)
        running = 0.0
        for i in range(0, n_tr, cfg["batch"]):
            idx = order[i:i + cfg["batch"]]
            ids, att = collate([enc["train"][j] for j in idx], tok.pad_token_id)
            labels = torch.tensor(y_tr[idx], dtype=torch.long).cuda()
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                loss = model(input_ids=ids.cuda(), attention_mask=att.cuda(), labels=labels).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * len(idx)
        val_scores = predict(model, enc["val"], tok.pad_token_id)
        ap = float(average_precision_score(y_va, val_scores))
        f1 = tb.tune_f1_threshold(y_va, val_scores)[1]
        history.append({"epoch": epoch, "train_loss": running / n_tr, "val_ap": ap, "val_best_f1": f1})
        print(f"[{key}] epoch {epoch}: train_loss {running / n_tr:.4f}  val AP {ap:.4f}  val best-F1 {f1:.4f}  ({time.time() - t0:.0f}s)")
        if ap > best_ap:
            best_ap, best_epoch, stale = ap, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if epoch >= MIN_EPOCHS and stale >= PATIENCE:
                print(f"[{key}] early stop after epoch {epoch} (best epoch {best_epoch})")
                break
    model.load_state_dict(best_state)
    return model, tok, enc, history, best_epoch, time.time() - t0

## 6. Train, freeze the threshold on val, then evaluate once

Evaluation reuses `train_baseline.py` (`evaluate_split`, `evaluate_tricky`, `top_errors`, ...), so the numbers are
directly comparable with the baselines: test (P/R/F1/FPR/confusion matrix, ROC-AUC, PR-AUC, per-source), tricky set
(counts per category), `xstest_safe` (FPR), the held-out `indirect_eval` set (metrics + recall split by annotated/bare insertions; evaluation only, never used for any choice), recall at the val-tuned FPR ≤ 1% threshold, and the 15 most confident val errors.

In [ ]:
results = {}
errors_md = [f"# Transformer val errors, top {tb.N_ERRORS} per direction\n", "Val only. Test errors are not inspected.\n"]

for variant, (col, train_file) in VARIANTS.items():
    data = {"train": raw[train_file].assign(label=raw[train_file][col]),
            **{s: raw[s].assign(label=raw[s][col]) for s in ["val", "test"]}}
    data["tricky"], data["xstest_safe"], data["indirect_eval"] = tricky, raw["xstest_safe"], raw["indirect_eval"]

    for key, cfg in MODELS.items():
        print(f"\n===== {variant} / {key} =====")
        model, tok, enc, history, best_epoch, secs = train_one(key, cfg, data, col)
        pad = tok.pad_token_id

        # thresholds on val only
        yv = data["val"]["label"].to_numpy()
        val_scores = predict(model, enc["val"], pad)
        thr, _ = tb.tune_f1_threshold(yv, val_scores)
        thr_fpr = tb.tune_fpr_threshold(yv, val_scores)

        # test / tricky / xstest are scored once, with the frozen thresholds
        test_scores = predict(model, enc["test"], pad)
        tr_scores = predict(model, enc["tricky"], pad)
        xs_scores = predict(model, enc["xstest_safe"], pad)
        res = {
            "description": f"{cfg['hf']} fine-tuned, head+tail {HEAD}+{TAIL}",
            "hyperparams": {**cfg, "head": HEAD, "tail": TAIL, "max_epochs": MAX_EPOCHS, "min_epochs": MIN_EPOCHS,
                            "patience": PATIENCE, "warmup_frac": WARMUP_FRAC, "weight_decay": WEIGHT_DECAY, "seed": SEED, "fp16": FP16},
            "n_params_m": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
            "history": history, "best_epoch": best_epoch, "train_seconds": round(secs),
            "threshold": thr, "threshold_fpr1pct": thr_fpr,
            "val": {**tb.rates(yv, val_scores >= thr), "pr_auc": float(average_precision_score(yv, val_scores))},
            "test": tb.evaluate_split(data["test"], test_scores, thr, thr_fpr),
            "tricky": tb.evaluate_tricky(tricky, tr_scores, thr),
            "indirect_eval": tb.evaluate_indirect(raw["indirect_eval"], predict(model, enc["indirect_eval"], pad), thr, thr_fpr),
            "xstest_safe": {"n": len(raw["xstest_safe"]), "false_positives": int((xs_scores >= thr).sum()),
                            "fpr": float((xs_scores >= thr).mean()),
                            "fpr_at_val_fpr_1pct_threshold": float((xs_scores >= thr_fpr).mean())},
        }
        results.setdefault(variant, {})[key] = res

        if variant in ("cleaned", "cleaned_aug"):
            fp, fn = tb.top_errors(data["val"], val_scores, thr)
            errors_md.append(f"## {key} [{variant}], val threshold {thr:.3f}\n")
            for title, frame in [("Most confident false positives", fp), ("Most confident false negatives", fn)]:
                errors_md.append(f"### {title} ({len(frame)})\n\n| score | source | text |\n|---|---|---|")
                errors_md += [f"| {r.score:.3f} | {r.source} | {tb.snippet(r.text).replace('|', '/')} |" for _, r in frame.iterrows()]
                errors_md.append("")

        ckpt = f"{OUT}/{variant}/{key}"
        model.save_pretrained(ckpt); tok.save_pretrained(ckpt)
        print(f"[{key}] saved {ckpt}")
        del model; torch.cuda.empty_cache()

## 7. Results table and files to download

In [ ]:
rows = []
for variant, models in results.items():
    for key, r in models.items():
        t, a = r["test"], r["test"]["at_val_fpr_1pct"]
        rows.append({"labels": variant, "model": key, "best_ep": r["best_epoch"], "thr": round(r["threshold"], 3),
                     "P": t["precision"], "R": t["recall"], "F1": t["f1"], "FPR": t["fpr"], "ROC-AUC": t["roc_auc"],
                     "R@FPR1%": a["recall"], "indirect F1": r["indirect_eval"]["f1"], "tricky": f'{r["tricky"]["_overall"]["correct"]}/{r["tricky"]["_overall"]["total"]}',
                     "xstest FP": f'{r["xstest_safe"]["false_positives"]}/{r["xstest_safe"]["n"]}', "min": round(r["train_seconds"] / 60, 1)})
table = pd.DataFrame(rows)
display(table.round(3))

meta = {"seed": SEED, "torch": torch.__version__, "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0),
        "rows": {k: len(v) for k, v in {**raw, "tricky": tricky}.items()}}
import transformers
meta["transformers"] = transformers.__version__
with open("metrics_transformers.json", "w") as f:
    json.dump({"meta": meta, "transformers": results}, f, indent=2)
with open("transformer_val_errors.md", "w", encoding="utf-8") as f:
    f.write("\n".join(errors_md))
print("wrote metrics_transformers.json and transformer_val_errors.md")

from google.colab import files
files.download("metrics_transformers.json"); files.download("transformer_val_errors.md")

## After this notebook (back in the repo)
1. Put both downloaded files in `ml/results/`. Do not edit the numbers.
2. If `USE_DRIVE = False`, the checkpoints in `/content/out` disappear when the runtime ends. Zip and download them
   (`!zip -qr ckpt.zip /content/out`) **before** closing the tab, or Step 5 has nothing to export.
3. Step 5 exports the checkpoints to ONNX (fp32 and int8) and applies the server/browser selection rules already written in `CLAUDE.md`.